In [ ]:
import os
import re
from collections import defaultdict
from pathlib import Path
import unicodedata
from collections import defaultdict

import numpy as np
import pandas as pd
import torch
from pypdf import PdfReader
from qdrant_client import QdrantClient, models
from qdrant_client.models import Distance, PointStruct, VectorParams
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from torch.nn import BCEWithLogitsLoss
from torch.nn.functional import sigmoid
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.utils.data import DataLoader, Dataset
from transformers import AutoTokenizer

from llama_index.core import Document
from llama_index.core.node_parser import SentenceSplitter, SemanticSplitterNodeParser
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
import re
import pandas as pd
from qdrant_client import models
import math
import numpy as np

from uuid import uuid5, NAMESPACE_URL

from qdrant_client.http.models import (
    Distance,
    VectorParams,
    PointStruct,
    PayloadSchemaType,
)

In [ ]:
from prometheus_eval import PrometheusEval
from prometheus_eval.prompts import ABSOLUTE_PROMPT
from prometheus_eval.litellm import LiteLLM

In [ ]:
print("torch:", torch.__version__)
print("cuda disponible:", torch.cuda.is_available())
print("torch cuda:", torch.version.cuda)
print("gpu count:", torch.cuda.device_count())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))

## 1 configurar el entorno

In [ ]:
client = QdrantClient(
    url="https://a5527f07-3110-4645-9e3c-7c8caea199dd.eu-central-1-0.aws.cloud.qdrant.io",
    api_key=os.getenv("QDRANT_API_KEY"),
    check_compatibility=False,
)

In [ ]:
# muchos modelos biomédicos de Hugging Face están pensados sobre todo para inglés y para literatura tipo PubMed/papers,
# no tanto para documentación regulatoria en españo

In [ ]:
# vamos a usar Es multilingüe, admite contexto largo y fue entrenado con adaptadores por tarea, así que para retrieval conviene usar el
# modo/tarea de búsqueda y no tratarlo como embedding genérico. Qdrant además documenta su uso y recuerda justamente ese punto de las tareas.

# otros 2 a probar son BAAI/bge-m3 (que es un poco peor)
# jinaai/jina-embeddings-v3
# Snowflake/snowflake-arctic-embed-l-v2.0 (que es mejor)

In [ ]:
# Directorio base del notebook
BASE_DIR = Path.cwd()

# Carpeta local donde están tus PDFs
PDF_FOLDER = BASE_DIR / "PDF.med"

# Nombre de la colección en Qdrant
COLLECTION_NAME = "rag_medico_baseline"

# Modelo de embeddings que usarás para indexar y consultar
EMBEDDING_MODEL_NAME = "BAAI/bge-m3"

# Dimensión del vector de ese modelo
VECTOR_SIZE = 1024

In [ ]:
# Este modelo convertirá cada chunk en un vector
model = HuggingFaceEmbedding(
    model_name=EMBEDDING_MODEL_NAME
)

print("Modelo de embeddings cargado:", EMBEDDING_MODEL_NAME)

## 2 Cargar PDFs
Aquí leemos los PDFs y extraemos el texto de cada página.

In [ ]:
# Buscar todos los PDFs de la carpeta
pdf_files = list(PDF_FOLDER.glob("*.pdf"))
print(f"PDFs encontrados en la carpeta: {len(pdf_files)}")

# Estructura donde guardaremos los documentos. Cada PDF tendrá:
# identificador, nombre de archivo, ruta, páginas con texto, texto completo unido
dataset_pdfs = []

In [ ]:
from pypdf import PdfReader

dataset_pdfs = []

for i, pdf_path in enumerate(pdf_files, start=1):
    reader = PdfReader(str(pdf_path))
    pages_data = []

    for page_num, page in enumerate(reader.pages, start=1):
        text = page.extract_text()

        # Guarda solo páginas con texto
        if text and text.strip():
            pages_data.append({
                "page_number": page_num,
                "text": text
            })

    # Une todo el texto extraído
    pdf_text = "\n".join(p["text"] for p in pages_data)

    # Payloads propios de la ingesta
    dataset_pdfs.append({
        "id": i,                          # id interno del documento
        "filename": pdf_path.name,        # nombre del archivo
        "filepath": str(pdf_path),        # ruta completa
        "stem": pdf_path.stem,            # nombre sin extensión
        "extension": pdf_path.suffix,     # extensión
        "pages_total": len(reader.pages), # páginas totales del PDF
        "pages_text": len(pages_data),    # páginas con texto extraído
        "pages": pages_data,              # texto por página
        "text": pdf_text,                 # texto completo unido
    })

In [ ]:
# - "Claves" muestra los campos guardados para cada PDF en dataset_pdfs.
# - "Archivo" es el nombre del PDF inspeccionado.
# - "Páginas totales" es el número total de páginas del PDF.
# - "Páginas con texto" indica en cuántas páginas se pudo extraer texto.
# - "Primera página con texto" es la primera página útil detectada.
# - "Preview" enseña un fragmento del texto extraído de esa página.
# - Que aparezcan saltos de línea, encabezados o "1 de 34" es normal: viene del propio PDF y se limpiará después.

print("Claves:", dataset_pdfs[0].keys())
print("Archivo:", dataset_pdfs[0]["filename"])
print("Páginas totales:", dataset_pdfs[0]["pages_total"])
print("Páginas con texto:", dataset_pdfs[0]["pages_text"])
print("Primera página con texto:", dataset_pdfs[0]["pages"][0]["page_number"])
print("Preview:", dataset_pdfs[0]["pages"][0]["text"][:300])

## 3 Limpieza de texto
normalizamos el texto extraído antes de segmentarlo.

In [ ]:
#### import re
import unicodedata

def clean_text(text: str) -> str:
    if not text:
        return ""

    # Normaliza unicode y saltos
    text = unicodedata.normalize("NFKC", text)
    text = text.replace("\r\n", "\n").replace("\r", "\n")

    # Une palabras cortadas por guion de salto de línea
    text = re.sub(r"(?<=\w)-\n(?=\w)", "", text)

    # Elimina numeración de página tipo "1 de 34"
    text = re.sub(r"(?mi)^\s*\d+\s+de\s+\d+\s*$", "", text)

    # Elimina cabeceras/pies muy frecuentes
    text = re.sub(r"(?mi)^\s*ficha técnica\s*$", "", text)
    text = re.sub(r"(?mi)^\s*prospecto:\s*información para el usuario\s*$", "", text)

    # Une líneas partidas dentro de párrafo
    text = re.sub(r"(?<!\n)\n(?!\n)", " ", text)

    # Mantiene separación de párrafos
    text = re.sub(r"\n{3,}", "\n\n", text)

    # Reduce espacios repetidos
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r" *\n *", "\n", text)

    return text.strip()

for item in dataset_pdfs:
    for page in item["pages"]:
        page["text_clean"] = clean_text(page["text"])

    item["text_clean"] = "\n\n".join(
        p["text_clean"] for p in item["pages"] if p["text_clean"].strip()
    )

In [ ]:
# Demostración breve de cómo queda tras la limpieza
print("Archivo:", dataset_pdfs[0]["filename"])
print("Página original:\n")
print(dataset_pdfs[0]["pages"][0]["text"][:500])

print("\n" + "-"*80 + "\n")

print("Página limpia:\n")
print(dataset_pdfs[0]["pages"][0]["text_clean"][:500])

## 4 Segmentación en bloques
cortar el texto en trozos con sentido, no por tamaño fijo.

Si se mete una página entera o chunks muy grandes, el embedding hace una especie de “promedio semántico” y una pregunta concreta puede no encontrar el fragmento preciso. Qdrant lo resume así: los chunks deben ser lo bastante pequeños para no diluir significado y lo bastante grandes para mantener contexto.

Para corpus médico es obligatorio.
De hecho, antes que semantic chunking, suele aportar más:

chunking por secciones/encabezados
metadata buena
hybrid retrieval
reranker

In [ ]:
def clean_block(block: str) -> str:
    # Reduce espacios repetidos
    block = re.sub(r"[ \t]+", " ", block)

    # Limpia espacios sobrantes
    return block.strip()


def split_long_block(block: str, max_chars: int = 1200):
    # Si no es largo, lo deja igual
    if len(block) <= max_chars:
        return [block]

    # Divide por frases
    sentences = re.split(r'(?<=[.!?])\s+', block)
    out = []
    current = ""

    for sent in sentences:
        sent = sent.strip()
        if not sent:
            continue

        if not current:
            current = sent
        elif len(current) + 1 + len(sent) <= max_chars:
            current += " " + sent
        else:
            out.append(current)
            current = sent

    if current:
        out.append(current)

    return out


def split_into_blocks(text: str, min_chars: int = 200, max_chars: int = 1200):
    # Separa por párrafos
    paragraphs = [clean_block(p) for p in text.split("\n\n")]
    paragraphs = [p for p in paragraphs if p]

    blocks = []
    current = ""

    for p in paragraphs:
        # Si el bloque actual es muy corto, sigue acumulando
        if not current:
            current = p
            continue

        if len(current) < min_chars:
            current += "\n\n" + p
            continue

        # Si añadir el siguiente párrafo lo hace demasiado grande, cierra bloque
        if len(current) + 2 + len(p) > max_chars:
            blocks.extend(split_long_block(current, max_chars=max_chars))
            current = p
        else:
            current += "\n\n" + p

    if current:
        blocks.extend(split_long_block(current, max_chars=max_chars))

    return blocks


# Aplicar segmentación a cada PDF
for item in dataset_pdfs:
    blocks = split_into_blocks(item["text_clean"], min_chars=200, max_chars=1200)

    item["blocks"] = [
        {
            "block_index": i,
            "text": block
        }
        for i, block in enumerate(blocks, start=1)
    ]

Esto Hace 3 cosas más útiles para RAG:

- Fusiona bloques cortos
Evita recuperar trozos pobres o ambiguos.

- Parte bloques largos
Evita embeddings demasiado “mezclados”.

- Mantiene un tamaño más estable
Los chunks se parecen más entre sí en granularidad.

In [ ]:
# Comprobación breve
for item in dataset_pdfs[:2]:
    print(item["filename"], "->", len(item["blocks"]), "bloques")
    if item["blocks"]:
        print(item["blocks"][0]["text"][:300])
        print("-" * 60)

## 5 Chunking final
Aquí convertimos los bloques en chunks del tamaño deseado
para después generar embeddings e indexarlos en Qdrant.

### 5.1 Helpers y deduplicación entre páginas

En este bloque se definen las funciones auxiliares que preparan el texto antes del chunking.

**Qué hace este bloque:**
- Normaliza saltos de línea y pequeñas inconsistencias de formato.
- Detecta inicios de sección (`4.1`, `4.2`, etc.) mediante expresiones regulares.
- Extrae el código de sección de cada fragmento.
- Calcula solapamientos exactos entre páginas consecutivas.
- Elimina repeticiones cuando una página arrastra texto ya presente en la anterior.

**Por qué es importante:**
En estos PDFs hay continuidad entre páginas y, a veces, parte del contenido aparece repetido en el borde entre una página y la siguiente. Si no se corrige aquí, el índice acaba con chunks redundantes y el retrieval recupera evidencias duplicadas.

In [ ]:
import re
from collections import defaultdict

SECTION_START_RE = re.compile(r'(?m)(?=^\s*\d{1,2}(?:\.\d+)?(?:\.)?\s+[A-ZÁÉÍÓÚÑ])')
HEADING_LINE_RE = re.compile(
    r'(?m)^\s*(?P<code>\d{1,2}(?:\.\d+)?)(?:\.)?\s+(?P<title>[^\n]+?)\s*$'
)
PAGE_MARKER_RE = re.compile(r'\[\[PAGE=(\d+)\]\]')

TITLES_42 = [
    "Administración concomitante con otros medicamentos",
    "Pacientes con insuficiencia hepática",
    "Pacientes con insuficiencia renal",
    "Pacientes de edad avanzada",
    "Forma de administración",
    "Población pediátrica",
    "Poblaciones especiales",
    "Posología",
    "Adultos",
]

SPECIAL_POP_42 = {
    "Pacientes con insuficiencia renal",
    "Pacientes con insuficiencia hepática",
    "Pacientes de edad avanzada",
}

RESTART_42 = {"Posología", "Adultos", "Población pediátrica", "Poblaciones especiales"}

HEADER_NOISE_42 = {
    "4.2", "4.2.", "posología", "y forma de administración", "posología y forma de administración"
}

def normalize_text_for_sections(text: str) -> str:
    if not text:
        return ""

    text = text.replace("\r\n", "\n").replace("\r", "\n")
    text = re.sub(
        r'(?<=[\.\):;])\s+(\d{1,2}(?:\.\d+)?(?:\.)?\s+[A-ZÁÉÍÓÚÑ])',
        r'\n\1',
        text
    )
    text = re.sub(r'\n{3,}', '\n\n', text)
    return text.strip()

def split_sections(text: str):
    text = normalize_text_for_sections(text)
    return [p.strip() for p in SECTION_START_RE.split(text) if p.strip()] or ([text] if text else [])

def build_heading_path(heading_code: str):
    if not heading_code:
        return []

    parts = heading_code.split(".")
    if len(parts) == 1:
        return [parts[0]]

    return [parts[0], heading_code]

def extract_heading_metadata(text: str):
    clean = normalize_text_for_sections(text)
    if not clean:
        return {
            "heading_path": [],
            "heading_leaf": None,
            "heading_title": None,
            "parent_section_code": None,
        }

    head_window = "\n".join(clean.splitlines()[:12])
    matches = list(HEADING_LINE_RE.finditer(head_window))

    if not matches:
        return {
            "heading_path": [],
            "heading_leaf": None,
            "heading_title": None,
            "parent_section_code": None,
        }

    first = matches[0]
    first_code = first.group("code").strip()
    first_title = first.group("title").strip()

    chosen_code = first_code
    chosen_title = first_title

    if "." not in first_code:
        for m in matches[1:]:
            candidate_code = m.group("code").strip()
            candidate_title = m.group("title").strip()

            if candidate_code.startswith(first_code + "."):
                chosen_code = candidate_code
                chosen_title = candidate_title
                break

    heading_path = build_heading_path(chosen_code)

    return {
        "heading_path": heading_path,
        "heading_leaf": chosen_code,
        "heading_title": chosen_title,
        "parent_section_code": heading_path[0] if heading_path else None,
    }

def get_section_code(text: str):
    return extract_heading_metadata(text).get("heading_leaf")

def norm_simple(text: str) -> str:
    return re.sub(r'\s+', ' ', text).strip(" :-").lower()

def exact_overlap(prev_text: str, curr_text: str, min_overlap: int = 80, max_overlap: int = 1200) -> int:
    if not prev_text or not curr_text:
        return 0
    max_len = min(len(prev_text), len(curr_text), max_overlap)
    for n in range(max_len, min_overlap - 1, -1):
        if prev_text[-n:] == curr_text[:n]:
            return n
    return 0

def dedup_adjacent_pages(item, min_overlap: int = 80, max_overlap: int = 1200):
    new_pages, prev_clean = [], ""

    for page in item.get("pages", []):
        page = dict(page)
        curr = page.get("text_clean", "").strip()

        overlap = exact_overlap(prev_clean, curr, min_overlap, max_overlap) if prev_clean and curr else 0
        if overlap:
            curr = curr[overlap:].lstrip()
            page["text_clean"] = curr

        new_pages.append(page)
        prev_clean = curr

    item = dict(item)
    item["pages"] = new_pages
    item["text_clean"] = "\n\n".join(
        p["text_clean"] for p in new_pages if p.get("text_clean", "").strip()
    )
    return item

### 5.2 Lógica específica para la sección 4.2

Este bloque implementa un tratamiento especial para la sección **4.2 (Posología y forma de administración)**.

**Qué hace este bloque:**
- Restaura cortes lógicos que en el PDF pueden venir mal unidos.
- Detecta subtítulos clínicamente relevantes dentro de 4.2.
- Separa subapartados como:
  - Posología
  - Poblaciones especiales
  - Pacientes con insuficiencia renal
  - Pacientes con insuficiencia hepática
  - Pacientes de edad avanzada
  - Forma de administración
- Conserva contexto cuando un subapartado depende de un encabezado padre.
- Divide fragmentos demasiado largos sin romper la estructura clínica.

**Por qué es importante:**
La sección 4.2 contiene información especialmente sensible en un RAG médico: dosis, ajuste en poblaciones especiales y forma de administración. Un chunking genérico aquí puede mezclar conceptos o cortar instrucciones relevantes.

In [ ]:
def restore_breaks_42(text: str) -> str:
    if not text:
        return ""

    text = text.replace("\r\n", "\n").replace("\r", "\n").strip()

    text = re.sub(
        r'^(4\.2(?:\.)?\s+Posología y forma de administración)\s+(?=Posología|Adultos|Población pediátrica|Poblaciones especiales)',
        r'\1\n',
        text,
        count=1,
        flags=re.IGNORECASE
    )

    cues = [
        r'Posología:?',
        r'Adultos',
        r'Población pediátrica',
        r'Poblaciones especiales',
        r'[•-]\s*Pacientes con insuficiencia renal:?',
        r'[•-]\s*Pacientes con insuficiencia hepática:?',
        r'[•-]\s*Pacientes de edad avanzada:?',
        r'Administración concomitante con otros medicamentos',
        r'Forma de administración:?'
    ]

    for cue in cues:
        text = re.sub(rf'\s+(?={cue})', '\n', text)

    text = re.sub(r'\s+(?=o\s+(?:Si|En)\b)', '\n', text)
    return re.sub(r'\n{2,}', '\n', text).strip()

def parse_42_title(line: str):
    raw = re.sub(r'^[•-]\s*', '', line).strip()

    for title in sorted(TITLES_42, key=len, reverse=True):
        if raw in {title, f"{title}:"}:
            return title, ""
        if raw.startswith(f"{title}:"):
            return title, raw[len(title)+1:].strip()
        if raw.startswith(f"{title} "):
            return title, raw[len(title):].strip()

    return raw.rstrip(":").strip(), ""

def is_42_title(line: str) -> bool:
    raw = re.sub(r'^[•-]\s*', '', line).strip()
    return any(
        raw == t or raw == f"{t}:" or raw.startswith(f"{t}:") or raw.startswith(f"{t} ")
        for t in TITLES_42
    )

def split_long_text(text: str, max_chars: int = 1200):
    if len(text) <= max_chars:
        return [text]

    parts = re.split(r'(?=\no\s+(?:Si|En)\b)', text)
    parts = [p.strip() for p in parts if p.strip()]

    if len(parts) == 1:
        parts = re.split(r'(?<=[.!?;:])\s+', text)
        parts = [p.strip() for p in parts if p.strip()]

    out, current = [], ""
    for part in parts:
        sep = "\n" if current and part.startswith("o ") else " "

        if not current:
            current = part
        elif len(current) + len(sep) + len(part) <= max_chars:
            current += sep + part
        else:
            out.append(current.strip())
            current = part

    if current:
        out.append(current.strip())

    return out

def split_section_42(section_text: str, heading_meta: dict = None, max_chars: int = 1200):
    heading_meta = heading_meta or {}
    base_meta = {
        "heading_path": heading_meta.get("heading_path", []),
        "heading_leaf": heading_meta.get("heading_leaf"),
        "heading_title": heading_meta.get("heading_title"),
        "parent_section_code": heading_meta.get("parent_section_code"),
    }

    lines = [ln.strip() for ln in restore_breaks_42(section_text).splitlines() if ln.strip()]
    if not lines:
        return []

    if lines[:3] == ["4.2.", "Posología", "y forma de administración"]:
        lines = ["4.2. Posología y forma de administración"] + lines[3:]

    if lines and lines[0].startswith(("4.2 ", "4.2.", "4.2. ")):
        lines = lines[1:]

    chunks, current_title, current_lines, parent_context = [], None, [], None

    def flush():
        nonlocal current_title, current_lines

        if not current_title and not current_lines:
            return

        body_lines = current_lines
        if current_title == "Posología":
            body_lines = [ln for ln in body_lines if norm_simple(ln) not in HEADER_NOISE_42]

        body = "\n".join(body_lines).strip()

        if current_title == "Posología":
            while True:
                prev = body
                for prefix in ["4.2.", "4.2", "Posología y forma de administración", "y forma de administración", "Posología"]:
                    if body.lower().startswith(prefix.lower()):
                        body = body[len(prefix):].strip(" :-\n")
                        break
                if body == prev:
                    break

        if current_title == "Posología" and not body:
            current_title, current_lines = None, []
            return

        parts = []
        if current_title in SPECIAL_POP_42 and parent_context:
            parts.append(parent_context)
        if current_title and current_title != "Posología":
            parts.append(current_title)

        prefix = "\n".join(parts).strip()

        if current_title == "Posología":
            full_text = f"Posología\n{body}"
        else:
            full_text = f"{prefix}\n{body}".strip() if prefix and body else (body or prefix)

        for piece in split_long_text(full_text, max_chars=max_chars):
            chunks.append({
                "text": piece,
                "subsection_title": current_title,
                **base_meta,
            })

        current_title, current_lines = None, []

    for line in lines:
        if is_42_title(line):
            title, inline_body = parse_42_title(line)

            if current_title == "Forma de administración" and title in RESTART_42:
                flush()
                break

            if title == "Poblaciones especiales":
                flush()
                parent_context = "Poblaciones especiales"
                current_title, current_lines = None, []
                continue

            flush()
            current_title = title
            current_lines = [inline_body] if inline_body else []
        else:
            if current_title is None:
                current_title = "Posología"
            current_lines.append(line)

    flush()
    return chunks

### 5.3 Split genérico del resto de secciones y merge final

Este bloque aplica el chunking al resto de secciones y realiza un postprocesado final sobre los chunks generados.

**Qué hace este bloque:**
- Divide las secciones que no son 4.2 usando una lógica más general.
- Intenta respetar bloques lógicos, viñetas y frases completas.
- Evita generar chunks excesivamente largos.
- Fusiona chunks demasiado cortos cuando pertenecen a la misma sección y subtítulo.
- Recalcula índices y `chunk_id` finales tras el merge.

**Por qué es importante:**
No todas las secciones necesitan una lógica tan específica como 4.2, pero sí conviene mantener cierta coherencia estructural. Además, el merge final ayuda a evitar fragmentos demasiado pequeños, que suelen empeorar embeddings, retrieval y reranking.

In [ ]:
def split_other_section(section_text: str, heading_meta: dict = None, max_chars: int = 1200):
    heading_meta = heading_meta or {}

    text = (section_text or "").strip()
    if not text:
        return []

    base_meta = {
        "heading_path": heading_meta.get("heading_path", []),
        "heading_leaf": heading_meta.get("heading_leaf"),
        "heading_title": heading_meta.get("heading_title"),
        "parent_section_code": heading_meta.get("parent_section_code"),
    }

    if len(text) <= max_chars:
        return [{
            "text": text,
            "subsection_title": None,
            **base_meta,
        }]

    lines = [ln.strip() for ln in text.splitlines() if ln.strip()]
    logical_blocks, current = [], ""

    def flush_block():
        nonlocal current
        if current.strip():
            logical_blocks.append(current.strip())
            current = ""

    for line in lines:
        is_main_bullet = line.startswith("") or bool(re.match(r"^-\s+[A-ZÁÉÍÓÚÑ]", line))
        is_sub_bullet = bool(re.match(r"^o\s+", line))

        if is_main_bullet:
            flush_block()
            current = line
        elif is_sub_bullet:
            current = f"{current}\n{line}" if current else line
        else:
            current = line if not current else f"{current} {line}"

    flush_block()

    chunks, current = [], ""
    for block in logical_blocks:
        if not current:
            current = block
        elif len(current) + 2 + len(block) <= max_chars:
            current += "\n\n" + block
        else:
            chunks.append({
                "text": current,
                "subsection_title": None,
                **base_meta,
            })
            current = block

    if current:
        chunks.append({
            "text": current,
            "subsection_title": None,
            **base_meta,
        })

    final_chunks = []
    for chunk in chunks:
        txt = chunk["text"]

        if len(txt) <= max_chars:
            final_chunks.append(chunk)
            continue

        current = ""
        for sent in re.split(r'(?<=[.!?;:])\s+', txt):
            sent = sent.strip()
            if not sent:
                continue

            if not current:
                current = sent
            elif len(current) + 1 + len(sent) <= max_chars:
                current += " " + sent
            else:
                final_chunks.append({
                    "text": current,
                    "subsection_title": None,
                    **base_meta,
                })
                current = sent

        if current:
            final_chunks.append({
                "text": current,
                "subsection_title": None,
                **base_meta,
            })

    return final_chunks

def split_section(section_text: str, section_code: str, heading_meta: dict = None, max_chars: int = 1200):
    if section_code == "4.2":
        return split_section_42(section_text, heading_meta=heading_meta, max_chars=max_chars)

    return split_other_section(section_text, heading_meta=heading_meta, max_chars=max_chars)

def merge_short_chunks(chunks, min_chars: int = 250, max_chars: int = 1400):
    merged = []

    for chunk in chunks:
        text = chunk.get("text", "").strip()
        if not text:
            continue

        if not merged:
            merged.append(chunk.copy())
            continue

        prev = merged[-1]
        same_place = (
            prev.get("doc_id") == chunk.get("doc_id")
            and prev.get("section_code") == chunk.get("section_code")
            and prev.get("heading_leaf") == chunk.get("heading_leaf")
            and prev.get("subsection_title") == chunk.get("subsection_title")
        )

        if same_place and len(text) < min_chars:
            combined = prev["text"].rstrip() + "\n\n" + text
            if len(combined) <= max_chars:
                prev["text"] = combined
                prev["char_len"] = len(combined)
                prev["word_len"] = len(combined.split())
                prev["page_end"] = chunk.get("page_end", prev.get("page_end"))
                prev["page_number"] = prev["page_start"] if prev.get("page_start") == prev.get("page_end") else None
                continue

        merged.append(chunk.copy())

    counters = defaultdict(int)
    for chunk in merged:
        stable_heading = chunk.get("heading_leaf") or chunk.get("section_code") or "na"
        key = (chunk.get("doc_id"), stable_heading)
        counters[key] += 1
        chunk["chunk_index"] = counters[key]
        chunk["chunk_id"] = f'{chunk.get("doc_id")}_{stable_heading}_{chunk["chunk_index"]}'

    return merged

### 5.4 Construcción final de chunks

Este bloque recorre todos los documentos ya limpiados y genera la colección final de chunks que se usará después para embeddings e indexación.

**Qué hace este bloque:**
- Aplica la deduplicación entre páginas a cada documento.
- Reconstruye el texto completo conservando marcadores de página.
- Separa el documento por secciones.
- Detecta el rango de páginas de cada sección.
- Aplica la estrategia de split adecuada:
  - lógica específica si la sección es 4.2
  - lógica general para el resto
- Añade metadatos importantes a cada chunk:
  - `filename`
  - `section_code`
  - `subsection_title`
  - `page_start`
  - `page_end`
  - `char_len`
  - `word_len`
- Ejecuta el merge final de chunks cortos.

**Resultado:**
Se obtiene `all_chunks`, que es la estructura final que alimentará la fase de embeddings, indexación y retrieval.

In [ ]:
if "dataset_pdfs" not in globals():
    raise ValueError("No existe `dataset_pdfs` en memoria. Ejecuta antes la carga/limpieza de PDFs.")

all_chunks = []

for raw_item in dataset_pdfs:
    item = dedup_adjacent_pages(raw_item, min_overlap=80, max_overlap=1200)

    doc_text = "\n\n".join(
        f"[[PAGE={p['page_number']}]]\n{p.get('text_clean', '').strip()}"
        for p in item.get("pages", [])
        if p.get("text_clean", "").strip()
    )

    for section in (split_sections(doc_text) or [doc_text]):
        pages = [int(p) for p in PAGE_MARKER_RE.findall(section)]
        section_text = PAGE_MARKER_RE.sub("", section).strip()

        if not section_text:
            continue

        heading_meta = extract_heading_metadata(section_text)
        section_code = heading_meta.get("heading_leaf") or get_section_code(section_text)

        page_start = min(pages) if pages else None
        page_end = max(pages) if pages else None

        split_chunks = split_section(
            section_text,
            section_code=section_code,
            heading_meta=heading_meta,
            max_chars=1200,
        )

        for idx, subchunk in enumerate(split_chunks, start=1):
            chunk_text = subchunk["text"].strip()
            if not chunk_text:
                continue

            subsection_title = subchunk.get("subsection_title")
            heading_leaf = subchunk.get("heading_leaf") or section_code
            heading_title = subchunk.get("heading_title") or heading_meta.get("heading_title")
            heading_path = subchunk.get("heading_path") or heading_meta.get("heading_path", [])
            parent_section_code = subchunk.get("parent_section_code") or heading_meta.get("parent_section_code")

            stable_heading = heading_leaf or section_code or "na"

            all_chunks.append({
                "chunk_id": f'{item["id"]}_{stable_heading}_{idx}',
                "doc_id": item["id"],
                "filename": item["filename"],

                "page_start": page_start,
                "page_end": page_end,
                "page_number": page_start if page_start == page_end else None,
                "pages_total": item["pages_total"],

                "section_code": section_code,
                "heading_path": heading_path,
                "heading_leaf": heading_leaf,
                "heading_title": heading_title,
                "parent_section_code": parent_section_code,

                "subsection_title": subsection_title,

                "chunk_index": idx,
                "char_len": len(chunk_text),
                "word_len": len(chunk_text.split()),
                "split_strategy": "section_heading+subsection" if subsection_title else "section_heading",
                "text": chunk_text,
            })

all_chunks = merge_short_chunks(all_chunks, min_chars=250, max_chars=1400)
print(f"Chunks generados: {len(all_chunks)}")

## 6 Generación de embeddings

En este bloque se transforma cada chunk semántico en un vector numérico usando el modelo de embeddings.

**Qué hace este bloque:**

- Selecciona solo los chunks con texto válido.
- Genera embeddings por lotes para evitar errores y controlar memoria.
- Verifica que cada texto tenga exactamente un embedding asociado.
- Comprueba que todos los embeddings tengan la misma dimensión.
- Inserta el embedding en el chunk original sin perder el resto de metadatos.

**Por qué es importante:**  
Si aquí falla la correspondencia entre chunks y embeddings, el índice de Qdrant queda inconsistente. En esta fase no se decide todavía el retrieval, pero sí se fija la representación vectorial sobre la que luego se buscará.

### 6.1 Generación robusta de embeddings por lotes

In [ ]:
def generate_embeddings_for_chunks(chunks, model, batch_size=64):
    """
    Genera embeddings solo para chunks con texto válido
    y los inserta de vuelta en la misma posición.
    """
    if batch_size <= 0:
        raise ValueError("batch_size debe ser mayor que 0")

    # 1) Selección robusta de chunks válidos
    # Solo se vectorizan chunks con texto no vacío.
    # Esto evita mandar al modelo entradas inválidas y mantiene
    # la posición original para reinyectar luego el embedding.
    candidates = []
    for idx, ch in enumerate(chunks):
        text = (ch.get("text") or "").strip()
        if not text:
            continue

        candidates.append({
            "idx": idx,
            "text": text,
        })

    print(f"Chunks totales: {len(chunks)}")
    print(f"Chunks a vectorizar: {len(candidates)}")

    if not candidates:
        raise ValueError("No hay chunks válidos para vectorizar")

    # 2) Generación por lotes controlados
    # Se procesa en batches para evitar sobrecarga y poder validar
    # lote a lote que no haya desajustes entre textos y embeddings.
    all_embeddings = []
    total_batches = math.ceil(len(candidates) / batch_size)

    for batch_num, start in enumerate(range(0, len(candidates), batch_size), start=1):
        batch = candidates[start:start + batch_size]
        batch_texts = [x["text"] for x in batch]

        print(f"Lote {batch_num}/{total_batches} | tamaño={len(batch_texts)}")

        batch_embeddings = model.get_text_embedding_batch(
            batch_texts,
            show_progress=False
        )

        if len(batch_embeddings) != len(batch_texts):
            raise ValueError(
                f"Desajuste en lote {batch_num}: "
                f"textos={len(batch_texts)} embeddings={len(batch_embeddings)}"
            )

        all_embeddings.extend(batch_embeddings)

    # 3) Validación global
    # Comprobamos que el número final de embeddings coincide
    # exactamente con el número de chunks vectorizados.
    if len(all_embeddings) != len(candidates):
        raise ValueError(
            f"Desajuste global: textos={len(candidates)} embeddings={len(all_embeddings)}"
        )

    dims = {len(e) for e in all_embeddings}
    if len(dims) != 1:
        raise ValueError(f"Dimensiones inconsistentes en embeddings: {dims}")

    embedding_dim = next(iter(dims))

    # 4) Inserción del embedding en el chunk original
    # Se convierte a float32 y se valida que no haya NaN o inf.
    for item, emb in zip(candidates, all_embeddings):
        emb = np.asarray(emb, dtype=np.float32)

        if np.isnan(emb).any() or np.isinf(emb).any():
            raise ValueError(f"Embedding inválido en chunk idx={item['idx']}")

        chunks[item["idx"]]["embedding"] = emb.tolist()

    print(f"Embeddings generados: {len(all_embeddings)}")
    print(f"Dimensión del embedding: {embedding_dim}")

    return chunks, embedding_dim


all_chunks, embedding_dim = generate_embeddings_for_chunks(
    all_chunks,
    model,
    batch_size=64
)

### 6.2 Verificación de consistencia de embeddings

Este bloque comprueba que la generación de embeddings ha sido correcta antes de pasar a indexación.

**Qué hace este bloque:**

- Cuenta cuántos chunks tienen embedding.
- Verifica que exista al menos uno.
- Comprueba que todos tengan la misma dimensión.
- Contrasta que todos los chunks con texto válido hayan recibido embedding.
- Muestra una inspección rápida del primer ejemplo.

**Por qué es importante:**  
Este chequeo evita que errores silenciosos pasen a Qdrant. Si aquí hay un desajuste, el problema no está en retrieval ni reranking, sino en la fase de vectorización.

In [ ]:
# 1) Chunks con texto válido
valid_text_chunks = [
    ch for ch in all_chunks
    if (ch.get("text") or "").strip()
]

# 2) Chunks que efectivamente tienen embedding
embedded_chunks = [
    ch for ch in all_chunks
    if "embedding" in ch and ch["embedding"] is not None
]

print("Chunks con texto válido:", len(valid_text_chunks))
print("Chunks con embedding:", len(embedded_chunks))

assert embedded_chunks, "No hay chunks con embedding"

# 3) Todos los chunks con texto deberían haber recibido embedding
assert len(valid_text_chunks) == len(embedded_chunks), (
    f"Desajuste: texto válido={len(valid_text_chunks)} vs embeddings={len(embedded_chunks)}"
)

# 4) Comprobación de dimensión uniforme
dims = {len(ch["embedding"]) for ch in embedded_chunks}
assert len(dims) == 1, f"Dimensiones inconsistentes: {dims}"

print("Dimensión embedding:", next(iter(dims)))
print("Primer chunk:", embedded_chunks[0]["chunk_id"])
print("Primer texto:", embedded_chunks[0]["text"][:200], "...")
print("Primeros 5 valores embedding:", embedded_chunks[0]["embedding"][:5])

## 7 Crear colección e indexar en Qdrant
guardamos cada chunk + embedding dentro de Qdrant

### 7.1 Clasificación y payload

Este bloque clasifica los documentos y construye el payload que se guardará junto a cada vector.

**Qué hace este bloque:**

- Asigna cada documento a una familia semántica.
- Detecta si el archivo corresponde a una formulación concreta.
- Normaliza campos numéricos.
- Genera un identificador estable para Qdrant.
- Construye el payload final de cada chunk.
- Propaga la nueva metadata estructural:
  - `heading_path`
  - `heading_path_str`
  - `heading_leaf`
  - `heading_title`
  - `parent_section_code`

**Por qué es importante:**  
El payload no solo sirve para trazabilidad. También define qué campos podrán usarse después en filtros, rescoring, evaluación y citas. Si aquí no entra la estructura jerárquica del chunk, luego no podrá explotarse en retrieval.

In [ ]:
REGULATORIOS = {
    "prospecto_medproduct.pdf",
    "medproduct. centro de información medicamentos aemps - cima.pdf",
    "100_20_2.5.pdf",
    "100_20_5.pdf",
    "100_20_10.pdf",
    "100_40_2.5.pdf",
    "100_40_5.pdf",
    "100_40_10.pdf",
}

EVIDENCIA = {
    "publicación estudio secure.pdf",
    "estudo pandora - rec versión española.pdf",
    "neptuno - juanatey 2022.pdf",
    "gaziano-etal-2025-cost-effectiveness-of-the-cv-polypill.pdf",
}

FORMULATIONS = {
    "100_20_2.5",
    "100_20_5",
    "100_20_10",
    "100_40_2.5",
    "100_40_5",
    "100_40_10",
}

def get_doc_family(filename: str) -> str:
    """
    Clasifica el documento en una familia útil para retrieval y análisis.
    """
    name = filename.lower()

    if name in REGULATORIOS:
        return "regulatorio"
    if name in EVIDENCIA:
        return "evidencia"
    return "otro"

def get_formulation(filename: str):
    """
    Extrae la formulación a partir del nombre del fichero cuando aplica.
    """
    stem = filename.replace(".pdf", "")
    return stem if stem in FORMULATIONS else None

def as_int_or_none(x):
    """
    Convierte a entero si es posible; en caso contrario devuelve None.
    """
    if x is None:
        return None
    try:
        return int(x)
    except Exception:
        return None

def make_point_id(chunk_id: str) -> str:
    """
    Genera un identificador estable para Qdrant a partir del chunk_id.
    """
    return str(uuid5(NAMESPACE_URL, chunk_id))

def build_payload(chunk: dict) -> dict:
    """
    Construye el payload del punto indexado.

    Además de la trazabilidad básica, propagamos la metadata estructural
    fina del chunk para poder usarla después en retrieval, evaluación y citas.
    """
    filename = chunk["filename"]

    return {
        "chunk_id": chunk["chunk_id"],
        "doc_id": chunk["doc_id"],
        "filename": filename,
        "family": get_doc_family(filename),
        "formulation": get_formulation(filename),
        "language": "es",

        "pages_total": as_int_or_none(chunk.get("pages_total")),
        "page_number": as_int_or_none(chunk.get("page_number")),
        "page_start": as_int_or_none(chunk.get("page_start")),
        "page_end": as_int_or_none(chunk.get("page_end")),

        # Metadata estructural
        "section_code": chunk.get("section_code"),
        "heading_path": chunk.get("heading_path") or [],
        "heading_path_str": " > ".join(chunk.get("heading_path") or []),
        "heading_leaf": chunk.get("heading_leaf"),
        "heading_title": chunk.get("heading_title"),
        "parent_section_code": chunk.get("parent_section_code"),

        # Metadata adicional del chunk
        "subsection_title": chunk.get("subsection_title"),
        "chunk_index": as_int_or_none(chunk.get("chunk_index")),
        "split_strategy": chunk.get("split_strategy"),

        "char_len": as_int_or_none(chunk.get("char_len")),
        "word_len": as_int_or_none(chunk.get("word_len")),

        # Texto completo del chunk
        "text": chunk["text"].strip(),
    }

### 7.2 Crear la colección en Qdrant

Este bloque recrea la colección vectorial usando la dimensión real del embedding.

**Qué hace este bloque:**

- Elimina la colección anterior si existía.
- Crea una nueva colección limpia.
- Define la dimensión del vector.
- Usa distancia coseno para similitud.

**Por qué es importante:**  
Si la dimensión no coincide con la del embedding real, la indexación fallará o quedará corrupta. Este paso asegura que la colección esté alineada con el modelo vectorial usado.

In [ ]:
# 7.2 Crear colección desde cero
client.recreate_collection(
    collection_name=COLLECTION_NAME,
    vectors_config=VectorParams(
        size=embedding_dim,
        distance=Distance.COSINE,
    ),
)

print(f"Colección '{COLLECTION_NAME}' recreada con dimensión {embedding_dim}")

### 7.3 Crear índices de metadatos

Este bloque crea índices de payload para que ciertos campos puedan usarse eficientemente en filtros exactos.

**Qué hace este bloque:**

- Define campos keyword para filtros exactos.
- Define campos enteros para filtros numéricos.
- Añade los nuevos campos estructurales al conjunto indexable:
  - `heading_path_str`
  - `heading_leaf`
  - `heading_title`
  - `parent_section_code`
- Crea los índices en Qdrant.

**Por qué es importante:**  
Si un campo está en el payload pero no tiene índice, puede seguir siendo visible, pero no será tan útil para filtros rápidos o reglas de retrieval. Aquí dejamos preparada la estructura para usar el heading fino en vez de depender solo de `section_code`.

In [ ]:
# 7.3 Crear índices de payload
keyword_fields = [
    "chunk_id",
    "doc_id",
    "filename",
    "family",
    "formulation",
    "language",
    "section_code",
    "heading_path_str",
    "heading_leaf",
    "heading_title",
    "parent_section_code",
    "subsection_title",
    "split_strategy",
]

integer_fields = [
    "pages_total",
    "page_number",
    "page_start",
    "page_end",
    "chunk_index",
    "char_len",
    "word_len",
]

# Índices para filtros de texto exacto
for field in keyword_fields:
    client.create_payload_index(
        collection_name=COLLECTION_NAME,
        field_name=field,
        field_schema=PayloadSchemaType.KEYWORD,
    )

# Índices para filtros numéricos
for field in integer_fields:
    client.create_payload_index(
        collection_name=COLLECTION_NAME,
        field_name=field,
        field_schema=PayloadSchemaType.INTEGER,
    )

print("Índices de payload creados")

### 7.4 Preparar los puntos a indexar

Este bloque transforma cada chunk válido en un `PointStruct` listo para insertarse en Qdrant.

**Qué hace este bloque:**

- Recorre todos los chunks generados.
- Descarta chunks vacíos o sin embedding.
- Valida que la dimensión del embedding sea correcta.
- Construye cada punto con:
  - id estable
  - vector
  - payload

**Por qué es importante:**  
Este es el último punto de control antes de indexar. Si aquí se cuela un embedding inválido o un payload mal construido, el error pasará al índice y será más difícil de detectar después.

In [ ]:
points = []

for chunk in all_chunks:
    text = (chunk.get("text") or "").strip()
    embedding = chunk.get("embedding")

    # Saltamos chunks vacíos o sin embedding
    if not text or embedding is None:
        continue

    # Validamos la dimensión antes de indexar
    if len(embedding) != embedding_dim:
        raise ValueError(
            f"Embedding inválido en {chunk['chunk_id']}: "
            f"{len(embedding)} != {embedding_dim}"
        )

    points.append(
        PointStruct(
            id=make_point_id(chunk["chunk_id"]),
            vector=embedding,
            payload=build_payload(chunk),
        )
    )

print("Puntos preparados:", len(points))

### 7.5 Indexar en Qdrant

Este bloque inserta los puntos en la colección por lotes.

**Qué hace este bloque:**

- Divide los puntos en batches.
- Inserta cada batch con `upsert`.
- Espera a que la operación termine antes de continuar.
- Informa del progreso de indexación.

**Por qué es importante:**  
La indexación por lotes reduce errores operativos y facilita detectar si un problema aparece en una fase concreta de la subida.

In [ ]:
batch_size = 128

for i in range(0, len(points), batch_size):
    batch = points[i:i + batch_size]
    client.upsert(
        collection_name=COLLECTION_NAME,
        points=batch,
        wait=True,
    )
    print(f"Lote {i // batch_size + 1}: {len(batch)} puntos insertados")

print("Indexación completada")

### 7.6 Comprobación rápida

Este bloque verifica que la colección existe y que los puntos han quedado indexados.

**Qué hace este bloque:**

- Recupera la información básica de la colección.
- Muestra el nombre de la colección.
- Muestra el número de puntos almacenados.
- Muestra el número de vectores indexados.

**Por qué es importante:**  
Es una comprobación mínima pero útil para confirmar que la fase de indexación ha terminado correctamente antes de pasar a retrieval y evaluación.

In [ ]:
collection_info = client.get_collection(COLLECTION_NAME)

print("Colección:", COLLECTION_NAME)
print("Puntos indexados:", collection_info.points_count)
print("Vectores indexados:", collection_info.indexed_vectors_count)

### 7.7 Búsqueda de prueba compatible

Este bloque ejecuta una búsqueda mínima sobre la colección recién indexada.

**Qué hace este bloque:**

- Toma el embedding de un chunk ya indexado.
- Lanza una consulta vectorial simple.
- Soporta clientes Qdrant con `query_points` o con `search`.
- Recupera los primeros resultados con payload.
- Muestra `score`, `chunk_id` y `filename`.

**Por qué es importante:**  
La colección puede existir y tener puntos, pero aun así fallar en consulta. Este bloque valida que el circuito mínimo de búsqueda vectorial funciona correctamente.

In [ ]:
test_vector = all_chunks[0]["embedding"]

if hasattr(client, "query_points"):
    response = client.query_points(
        collection_name=COLLECTION_NAME,
        query=test_vector,
        limit=3,
        with_payload=True,
        with_vectors=False,
    )
    hits = response.points
elif hasattr(client, "search"):
    hits = client.search(
        collection_name=COLLECTION_NAME,
        query_vector=test_vector,
        limit=3,
    )
else:
    raise AttributeError("Tu cliente de Qdrant no expone ni query_points ni search")

print("Hits devueltos:", len(hits))
for hit in hits:
    payload = hit.payload if hasattr(hit, "payload") else {}
    score = hit.score if hasattr(hit, "score") else None
    print(score, payload.get("chunk_id"), payload.get("filename"))

## 8 Búsqueda y evaluación del retrieval

En este bloque se define el retrieval baseline, se construyen filtros a partir de la query y se evalúa el comportamiento del buscador sobre un banco de preguntas.

**Qué hace este bloque:**

- Configura los parámetros base de búsqueda.
- Detecta filtros útiles desde la query:
  - familia documental
  - formulación
  - heading estructural
  - subapartado clínico
  - documento explícito
- Ejecuta retrieval baseline con relajación progresiva de filtros.
- Permite inspección manual de resultados.
- Define un banco de evaluación reproducible.
- Calcula métricas de retrieval:
  - `hit@1`
  - `hit@3`
  - `hit@5`
  - `MRR`

**Por qué es importante:**  
Este bloque es donde se comprueba si la nueva granularidad estructural realmente mejora el acceso a la evidencia correcta. El cambio clave aquí es pasar de evaluar por `section_code` a evaluar por `heading_leaf`.

### 8.1 Configuración base de la búsqueda

Este bloque define los parámetros globales del retrieval y algunos helpers para localizar documentos conocidos dentro del corpus.

**Qué hace este bloque:**

- Fija `TOP_K` y el tamaño del pool de candidatos.
- Construye el listado de nombres de archivo conocidos.
- Define una normalización ligera para comparar texto.
- Localiza automáticamente los documentos canónicos de:
  - `SECURE`
  - `PANDORA`
  - `NEPTUNO`
- Define una función compatible para consultar Qdrant con o sin `query_points`.

**Por qué es importante:**  
Antes de aplicar filtros o evaluar, conviene dejar fijo el marco de búsqueda y los documentos de referencia que luego se usarán en las queries de evaluación.

In [ ]:
TOP_K = 3
CANDIDATE_POOL = 12

KNOWN_FILENAMES = sorted({c["filename"] for c in all_chunks})

def norm_text(x):
    return re.sub(r"\s+", " ", str(x or "").lower()).strip()

def find_filename_contains(text):
    t = norm_text(text)
    for f in KNOWN_FILENAMES:
        if t in norm_text(f):
            return f
    return None

CANONICAL_STUDIES = {
    "secure": find_filename_contains("secure"),
    "pandora": find_filename_contains("pandora"),
    "neptuno": find_filename_contains("neptuno"),
}

def run_qdrant_query(query_vector, top_k, qfilter=None):
    kwargs = {
        "collection_name": COLLECTION_NAME,
        "limit": top_k,
        "with_payload": True,
        "with_vectors": False,
    }

    try:
        if qfilter is not None:
            return client.query_points(
                query=query_vector,
                query_filter=qfilter,
                **kwargs
            ).points

        return client.query_points(
            query=query_vector,
            **kwargs
        ).points

    except TypeError:
        if qfilter is not None:
            return client.search(
                query_vector=query_vector,
                query_filter=qfilter,
                **kwargs
            )

        return client.search(
            query_vector=query_vector,
            **kwargs
        )

### 8.2 Detección de filtros desde la query

Este bloque traduce la intención de la query a filtros estructurados para retrieval.

**Qué hace este bloque:**

- Detecta formulaciones del tipo `100_20_5`.
- Detecta referencias explícitas a headings regulatorios como `4.1`, `4.2`, `4.3`.
- Mapea temas regulatorios frecuentes a headings concretos.
- Detecta subapartados clínicos relevantes dentro de `4.2`.
- Detecta menciones explícitas a documentos concretos.
- Infiere la familia documental cuando la intención de la query es clara.
- Construye el diccionario final de filtros.
- Construye el filtro de Qdrant.

**Cambio principal respecto a la versión anterior:**  
Aquí dejamos de usar `section_code` como señal principal y pasamos a usar `heading_leaf`, que representa mejor la unidad estructural real del chunk.

**Por qué es importante:**  
Si una query pide indicaciones terapéuticas, lo correcto es filtrar por `4.1`, no por el padre `4`. Este bloque es el punto donde esa decisión se vuelve efectiva.

In [ ]:
FORMULATION_RE = re.compile(r"\b100[\s_/-]?(20|40)[\s_/-]?(2[.,]5|5|10)\b", re.I)
HEADING_RE = re.compile(r"\b(?:secci[oó]n|apartado)?\s*(4(?:\.\d+)?)\b", re.I)

SUBSECTION_RULES = [
    ("Población pediátrica", [r"pedi[aá]tric", r"niñ", r"adolesc"]),
    ("Pacientes con insuficiencia renal", [r"insuficiencia renal", r"\brenal\b"]),
    ("Pacientes con insuficiencia hepática", [r"insuficiencia hep[aá]tica", r"\bhep[aá]tic"]),
    ("Pacientes de edad avanzada", [r"edad avanzada", r"ancian", r"geri[aá]tric"]),
    ("Forma de administración", [r"forma de administr", r"c[oó]mo administrar"]),
]

# Mapeo estructural estable para documentos regulatorios.
# La clave ya no es section_code "grueso", sino heading_leaf "fino".
HEADING_TOPIC_RULES = [
    ("4.1", [r"indicaci[oó]n", r"indicaciones terap[eé]uticas"]),
    ("4.2", [r"posolog", r"\bdosis\b", r"administr", r"pedi[aá]tric", r"renal", r"hep[aá]tic", r"edad avanzada"]),
    ("4.3", [r"contraindic"]),
    ("4.4", [r"advertencias?", r"precauciones?"]),
    ("4.5", [r"interacciones?", r"concomitante"]),
]

EXPLICIT_DOC_RULES = [
    (r"\bsecure\b", CANONICAL_STUDIES.get("secure")),
    (r"\bpandora\b", CANONICAL_STUDIES.get("pandora")),
    (r"\bneptuno\b", CANONICAL_STUDIES.get("neptuno")),
    (r"\bprospecto\b", find_filename_contains("prospecto")),
    (r"\bcima\b", find_filename_contains("cima")),
    (r"\baemps\b", find_filename_contains("cima") or find_filename_contains("aemps")),
    (r"coste[- ]efectividad|cost effectiveness", find_filename_contains("cost-effectiveness")),
]

def infer_formulation(query):
    m = FORMULATION_RE.search(query or "")
    if not m:
        return None
    return f"100_{m.group(1)}_{m.group(2).replace(',', '.')}"

def infer_heading_leaf(query):
    """
    Devuelve el heading regulatorio fino esperado.
    Ejemplos:
    - query explícita "apartado 4.3" -> 4.3
    - query temática "contraindicaciones" -> 4.3
    """
    m = HEADING_RE.search(query or "")
    if m:
        return m.group(1)

    q = norm_text(query)
    for heading_leaf, patterns in HEADING_TOPIC_RULES:
        if any(re.search(p, q, re.I) for p in patterns):
            return heading_leaf

    return None

def infer_subsection(query):
    q = norm_text(query)
    for title, patterns in SUBSECTION_RULES:
        if any(re.search(p, q, re.I) for p in patterns):
            return title
    return None

def infer_filename(query):
    q = norm_text(query)
    for pattern, filename in EXPLICIT_DOC_RULES:
        if filename and re.search(pattern, q, re.I):
            return filename
    return None

def infer_family(query, filename=None):
    q = norm_text(query)

    if filename in {
        CANONICAL_STUDIES.get("secure"),
        CANONICAL_STUDIES.get("pandora"),
        CANONICAL_STUDIES.get("neptuno"),
    }:
        return "evidencia"

    if filename and ("cost-effectiveness" in norm_text(filename)):
        return None

    if re.search(r"ficha t[eé]cnica|prospecto|cima|aemps", q, re.I):
        return "regulatorio"

    if re.search(r"estudio|ensayo|publicaci[oó]n|paper|art[ií]culo", q, re.I):
        return "evidencia"

    return None

def build_filters(query):
    filename = infer_filename(query)

    filters_dict = {
        "family": infer_family(query, filename=filename),
        "formulation": infer_formulation(query),
        "heading_leaf": infer_heading_leaf(query),
        "subsection_title": infer_subsection(query),
        "filename": filename,
    }

    return filters_dict

def build_qdrant_filter(filters_dict):
    must = []

    for key in ["family", "formulation", "heading_leaf", "subsection_title", "filename"]:
        value = filters_dict.get(key)
        if value:
            must.append(
                models.FieldCondition(
                    key=key,
                    match=models.MatchValue(value=value)
                )
            )

    return models.Filter(must=must) if must else None

### 8.3 Retrieval baseline con relajación progresiva

Este bloque implementa el retrieval baseline y la lógica de relajación de filtros.

**Qué hace este bloque:**

- Define un orden de relajación progresiva de filtros.
- Ejecuta retrieval con distintos niveles de restricción.
- Recupera candidatos desde Qdrant.
- Calcula un bonus ligero para desempatar casos frecuentes.
- Conserva solo la mejor versión de cada chunk.
- Devuelve los resultados finales ordenados.

**Cambio principal respecto a la versión anterior:**  
La relajación ahora elimina antes `heading_leaf` que otros filtros más generales. Esto hace visible cuándo el hit correcto depende realmente del heading fino.

**Por qué es importante:**  
Así puedes distinguir mejor entre:
- fallo de recall,
- fallo de filtro demasiado estricto,
- y fallo de ranking real.

In [ ]:
def relaxation_plans(base_filters):
    order = ["subsection_title", "heading_leaf", "formulation", "family", "filename"]
    plans = [{"stage": "strict", "filters": dict(base_filters)}]

    current = dict(base_filters)
    for key in order:
        if current.get(key):
            current = dict(current)
            current[key] = None
            plans.append({"stage": f"drop_{key}", "filters": dict(current)})

    if plans[-1]["filters"]:
        plans.append({"stage": "no_filters", "filters": {}})

    return plans


def chunk_bonus(row, query):
    bonus = 0.0

    q_heading = infer_heading_leaf(query)
    q_subsection = infer_subsection(query)
    q_filename = infer_filename(query)

    if q_heading and row.get("heading_leaf") == q_heading:
        bonus += 0.05
    if q_subsection and row.get("subsection_title") == q_subsection:
        bonus += 0.03
    if q_filename and row.get("filename") == q_filename:
        bonus += 0.05

    return bonus

In [ ]:
def search_chunks(query, top_k=TOP_K, candidate_pool=CANDIDATE_POOL):
    query_vector = model.get_query_embedding(query)
    base_filters = build_filters(query)
    plans = relaxation_plans(base_filters)

    for plan in plans:
        hits = run_qdrant_query(
            query_vector=query_vector,
            top_k=candidate_pool,
            qfilter=build_qdrant_filter(plan["filters"])
        )

        rows = []
        for hit in hits:
            payload = hit.payload or {}

            row = {
                "score": float(hit.score),
                "stage": plan["stage"],
                "filename": payload.get("filename"),
                "family": payload.get("family"),
                "formulation": payload.get("formulation"),
                "heading_leaf": payload.get("heading_leaf"),
                "heading_title": payload.get("heading_title"),
                "parent_section_code": payload.get("parent_section_code"),
                "section_code": payload.get("section_code"),
                "subsection_title": payload.get("subsection_title"),
                "page_start": payload.get("page_start"),
                "page_end": payload.get("page_end"),
                "chunk_id": payload.get("chunk_id"),
                "text": payload.get("text", "") or "",
            }

            row["bonus"] = chunk_bonus(row, query)
            row["final_score"] = row["score"] + row["bonus"]
            rows.append(row)

        if rows:
            ranked = sorted(
                rows,
                key=lambda x: (x["final_score"], x["score"]),
                reverse=True
            )[:top_k]

            for i, row in enumerate(ranked, start=1):
                row["rank"] = i

            return ranked

    return []

### 8.4 Inspección manual de una query

Este bloque permite revisar visualmente qué devuelve el retrieval para una pregunta concreta.

**Qué hace este bloque:**

- Ejecuta `search_chunks()` sobre una query.
- Muestra los primeros resultados con score y etapa de relajación.
- Enseña el `heading_leaf` y el `heading_title`.
- Incluye una vista previa del texto recuperado.

**Por qué es importante:**  
Es la forma más rápida de comprobar si el sistema está cayendo en el apartado correcto. En esta iteración interesa especialmente revisar si preguntas regulatorias aterrizan en `4.1`, `4.2`, `4.3`, etc.

In [ ]:
def inspect_query(query, top_k=5, preview_chars=160):
    hits = search_chunks(query, top_k=top_k)

    df = pd.DataFrame([
        {
            "rank": h["rank"],
            "score": round(h["score"], 4),
            "bonus": round(h["bonus"], 4),
            "final_score": round(h["final_score"], 4),
            "stage": h["stage"],
            "filename": h["filename"],
            "heading_leaf": h["heading_leaf"],
            "heading_title": h["heading_title"],
            "subsection_title": h["subsection_title"],
            "preview": (h["text"] or "")[:preview_chars].replace("\n", " "),
        }
        for h in hits
    ])

    display(df)
    return hits

### 8.5 Banco de evaluación ampliado

Este bloque define el conjunto de queries con el que se medirá el retrieval baseline.

**Qué hace este bloque:**

- Localiza subconjuntos de documentos útiles para evaluación.
- Construye listas de archivos regulatorios.
- Construye listas de archivos de coste-efectividad.
- Define queries regulatorias, de evidencia y de economía.
- Sustituye `expected_section_code` por `expected_heading_leaf` en los casos regulatorios.

**Por qué es importante:**  
La evaluación tiene que medir la granularidad que realmente quieres optimizar. Si el chunk correcto es `4.1`, la evaluación también tiene que exigir `4.1`, no solo `4`.

In [ ]:
def find_filenames_contains_all(*terms):
    terms = [norm_text(t) for t in terms if t]
    out = []
    for f in KNOWN_FILENAMES:
        nf = norm_text(f)
        if all(t in nf for t in terms):
            out.append(f)
    return out

REGULATORY_FILENAMES = sorted(set(
    find_filenames_contains_all("medproduct") +
    [f for f in KNOWN_FILENAMES if re.search(r"^100_(20|40)_(2\.5|5|10)\.pdf$", f)]
))

COST_EFFECTIVENESS_FILENAMES = find_filenames_contains_all("cost-effectiveness")

EVAL_QUERIES = [
    {
        "id": "indicacion",
        "category": "regulatorio",
        "query": "¿Cuál es la indicación terapéutica de MedProduct?",
        "expected_family": "regulatorio",
        "expected_heading_leaf": "4.1",
    },
    {
        "id": "posologia",
        "category": "regulatorio",
        "query": "¿Qué dice MedProduct sobre la posología?",
        "expected_family": "regulatorio",
        "expected_heading_leaf": "4.2",
    },
    {
        "id": "pediatrica",
        "category": "regulatorio",
        "query": "¿Qué dice MedProduct sobre población pediátrica?",
        "expected_family": "regulatorio",
        "expected_heading_leaf": "4.2",
        "expected_subsection_title": "Población pediátrica",
    },
    {
        "id": "renal",
        "category": "regulatorio",
        "query": "¿Qué indica MedProduct en pacientes con insuficiencia renal?",
        "expected_family": "regulatorio",
        "expected_heading_leaf": "4.2",
        "expected_subsection_title": "Pacientes con insuficiencia renal",
    },
    {
        "id": "contraindicaciones",
        "category": "regulatorio",
        "query": "¿Qué contraindicaciones tiene MedProduct?",
        "expected_family": "regulatorio",
        "expected_heading_leaf": "4.3",
    },
    {
        "id": "secure_mace",
        "category": "evidencia",
        "query": "¿Qué resultado mostró SECURE sobre eventos cardiovasculares mayores?",
        "expected_filename": CANONICAL_STUDIES["secure"],
    },
    {
        "id": "secure_adherencia",
        "category": "evidencia",
        "query": "¿Qué mostró SECURE sobre la adherencia al tratamiento?",
        "expected_filename": CANONICAL_STUDIES["secure"],
    },
    {
        "id": "pandora_conclusion",
        "category": "evidencia",
        "query": "¿Qué concluye PANDORA?",
        "expected_filename": CANONICAL_STUDIES["pandora"],
    },
    {
        "id": "pandora_formulacion",
        "category": "evidencia",
        "query": "¿Qué formulación fue la más usada en PANDORA?",
        "expected_filename": CANONICAL_STUDIES["pandora"],
    },
    {
        "id": "neptuno_mace",
        "category": "evidencia",
        "query": "¿Qué muestra NEPTUNO sobre recurrencia de MACE?",
        "expected_filename": CANONICAL_STUDIES["neptuno"],
    },
    {
        "id": "neptuno_persistencia",
        "category": "evidencia",
        "query": "¿Qué muestra NEPTUNO sobre persistencia terapéutica?",
        "expected_filename": CANONICAL_STUDIES["neptuno"],
    },
    {
        "id": "coste_efectividad",
        "category": "economia",
        "query": "¿Qué concluye el estudio de coste-efectividad sobre la CV-polypill en España?",
        "expected_any_filename": COST_EFFECTIVENESS_FILENAMES,
    },
]

### 8.6 Reglas de relevancia y métricas

Este bloque define cuándo un hit se considera relevante y cómo se calculan las métricas del retrieval.

**Qué hace este bloque:**

- Comprueba si el hit pertenece al documento esperado.
- Comprueba si pertenece a la familia esperada.
- Comprueba si cae en el `heading_leaf` esperado.
- Comprueba el subapartado clínico cuando aplica.
- Calcula el primer rango relevante.
- Genera `hit@1`, `hit@3`, `hit@5` y `MRR`.
- Resume resultados globales y por categoría.

**Cambio principal respecto a la versión anterior:**  
La relevancia regulatoria pasa a evaluarse con `heading_leaf` en vez de `section_code`.

**Por qué es importante:**  
Así la métrica ya refleja el nivel de precisión estructural que quieres conseguir en el corpus regulatorio.

In [ ]:
def is_relevant(hit, spec):
    if spec.get("expected_filename") and hit.get("filename") != spec["expected_filename"]:
        return False

    if spec.get("expected_any_filename") and hit.get("filename") not in spec["expected_any_filename"]:
        return False

    if spec.get("expected_family") and hit.get("family") != spec["expected_family"]:
        return False

    if spec.get("expected_heading_leaf") and hit.get("heading_leaf") != spec["expected_heading_leaf"]:
        return False

    if spec.get("expected_subsection_title") and hit.get("subsection_title") != spec["expected_subsection_title"]:
        return False

    return True

def first_relevant_rank(hits, spec):
    for i, hit in enumerate(hits, start=1):
        if is_relevant(hit, spec):
            return i
    return None

def evaluate_retrieval(eval_queries, top_k=5):
    rows = []

    for spec in eval_queries:
        hits = search_chunks(spec["query"], top_k=top_k)
        rr = first_relevant_rank(hits, spec)

        rows.append({
            "id": spec["id"],
            "category": spec["category"],
            "query": spec["query"],
            "hit@1": int(rr == 1),
            "hit@3": int(rr is not None and rr <= 3),
            "hit@5": int(rr is not None and rr <= 5),
            "mrr": 0.0 if rr is None else round(1 / rr, 3),
            "first_rel_rank": rr,
            "top1_file": hits[0]["filename"] if hits else None,
            "top1_stage": hits[0]["stage"] if hits else None,
            "top1_heading_leaf": hits[0]["heading_leaf"] if hits else None,
            "top1_heading_title": hits[0]["heading_title"] if hits else None,
            "top1_subsection": hits[0]["subsection_title"] if hits else None,
        })

    eval_df = pd.DataFrame(rows)

    overall_df = pd.DataFrame([{
        "n_queries": len(eval_df),
        "hit@1": round(eval_df["hit@1"].mean(), 3),
        "hit@3": round(eval_df["hit@3"].mean(), 3),
        "hit@5": round(eval_df["hit@5"].mean(), 3),
        "mrr": round(eval_df["mrr"].mean(), 3),
    }])

    by_category_df = (
        eval_df
        .groupby("category", as_index=False)
        .agg(
            n_queries=("id", "count"),
            hit_at_1=("hit@1", "mean"),
            hit_at_3=("hit@3", "mean"),
            hit_at_5=("hit@5", "mean"),
            mrr=("mrr", "mean"),
        )
        .round(3)
    )

    display(eval_df)
    display(overall_df)
    display(by_category_df)

    return eval_df, overall_df, by_category_df

### 8.7 Ejecutar la evaluación y guardar resultados

Este bloque ejecuta la evaluación del retrieval sobre el banco definido y conserva una copia de los resultados.

**Qué hace este bloque:**

- Ejecuta `evaluate_retrieval`.
- Guarda los dataframes de evaluación.
- Crea copias independientes para análisis posterior.

**Por qué es importante:**  
Deja un punto de referencia estable para comparar cambios futuros en retrieval y reranking.

In [ ]:
eval_df_v2, overall_df_v2, by_category_df_v2 = evaluate_retrieval(EVAL_QUERIES, top_k=5)

eval_df_v2 = eval_df_v2.copy()
overall_df_v2 = overall_df_v2.copy()
by_category_df_v2 = by_category_df_v2.copy()

In [ ]:
# Con la configuración base, el sistema acierta el resultado correcto en primera posición en aproximadamente 
# dos de cada tres preguntas

### 8.8 Revisar errores prioritarios

Este bloque selecciona los casos donde el retrieval todavía falla o donde el primer relevante no aparece en la primera posición.

**Qué hace este bloque:**

- Filtra queries sin relevante en top-3.
- Filtra queries sin relevante recuperado.
- Filtra queries donde el primer relevante no está en rango 1.
- Ordena los fallos para revisión.

**Por qué es importante:**  
Permite concentrar la inspección manual en los errores que realmente limitan el rendimiento del baseline.

In [ ]:
failures_df_v2 = eval_df_v2[
    (eval_df_v2["hit@3"] == 0) |
    (eval_df_v2["first_rel_rank"].isna()) |
    (eval_df_v2["first_rel_rank"] > 1)
].copy()

display(
    failures_df_v2.sort_values(
        ["category", "hit@3", "first_rel_rank"],
        na_position="last"
    )
)

### 8.9 Inspección manual de los fallos

Este bloque recorre las queries problemáticas y muestra manualmente los resultados devueltos por el retrieval.

**Qué hace este bloque:**

- Itera sobre las queries fallidas.
- Imprime cada pregunta.
- Ejecuta `inspect_query()` para revisar los hits recuperados.

**Por qué es importante:**  
Sirve para ver si el fallo es de:
- recall,
- filtro,
- ranking,
- o granularidad estructural.

In [ ]:
for q in failures_df_v2["query"].tolist():
    print("=" * 120)
    print(q)
    inspect_query(q, top_k=5)

### 8.10 Comparar métricas entre versiones

Este bloque define una utilidad para comparar dos resúmenes de métricas globales.

**Qué hace este bloque:**

- Toma dos dataframes resumen.
- Extrae `hit@1`, `hit@3`, `hit@5` y `MRR`.
- Calcula el delta entre ambas versiones.

**Por qué es importante:**  
Permite medir de forma limpia si los cambios en chunking, filtros o ranking realmente mejoran el retrieval.

In [ ]:
def compare_overall_metrics(before_df, after_df, before_name="antes", after_name="después"):
    metrics = ["hit@1", "hit@3", "hit@5", "mrr"]
    rows = []

    before_row = before_df.iloc[0].to_dict()
    after_row = after_df.iloc[0].to_dict()

    for m in metrics:
        rows.append({
            "metric": m,
            before_name: before_row[m],
            after_name: after_row[m],
            "delta": round(after_row[m] - before_row[m], 3),
        })

    return pd.DataFrame(rows)

# Ejemplo de uso:
# compare_overall_metrics(overall_df_v1, overall_df_v2)

## 9 reranking

### 9.1 Cargar el modelo de reranking

En este bloque se carga un `CrossEncoder` para hacer reranking. A diferencia del retrieval vectorial, que compara embeddings ya calculados, el reranker lee directamente la pareja **pregunta + chunk** y estima de forma más fina qué fragmento responde mejor a la query.

Se intenta usar primero un modelo multilingüe (`BAAI/bge-reranker-v2-m3`) y, si falla, se recurre a un modelo base como alternativa. La idea es probar si una segunda capa de ranking mejora el orden de los resultados recuperados por el baseline.

In [ ]:
# Si falta el paquete, ejecutar una vez:
# %pip install -q sentence-transformers

from sentence_transformers import CrossEncoder
import numpy as np

RERANKER_MODEL_NAME = "BAAI/bge-reranker-v2-m3"

# Carga del reranker
try:
    reranker = CrossEncoder(
        RERANKER_MODEL_NAME,
        max_length=512,
        trust_remote_code=True,
    )
except Exception:
    # Fallback simple si el modelo multilingüe falla
    RERANKER_MODEL_NAME = "BAAI/bge-reranker-base"
    reranker = CrossEncoder(
        RERANKER_MODEL_NAME,
        max_length=512,
    )

print("Reranker cargado:", RERANKER_MODEL_NAME)

### 9.2 Preparar la lógica de reranking

Aquí se simplifica el reranking para que el experimento sea más limpio.

**Qué hace este bloque:**
- define cuántos candidatos se recuperan antes de rerank
- construye el texto que verá el `CrossEncoder`
- calcula un `rerank_score` para cada candidato
- reordena los resultados usando directamente ese score

**Objetivo:**
Comprobar si el `CrossEncoder` mejora el orden de los candidatos por sí mismo, sin apoyarse en una combinación manual de scores.

In [ ]:
# Parámetros del reranking
RERANK_CANDIDATES = 12
RERANK_TOP_K = 3

def build_reranker_passage(hit):
    """Construye el texto que verá el reranker."""
    parts = []

    if hit.get("filename"):
        parts.append(f"Documento: {hit['filename']}")
    if hit.get("section_code"):
        parts.append(f"Sección: {hit['section_code']}")
    if hit.get("subsection_title"):
        parts.append(f"Subsección: {hit['subsection_title']}")

    parts.append(hit.get("text", "") or "")
    return "\n".join(parts).strip()

def rerank_hits(query, hits):
    """Reordena candidatos usando el score del CrossEncoder."""
    if not hits:
        return []

    pairs = [(query, build_reranker_passage(hit)) for hit in hits]

    rerank_scores = reranker.predict(
        pairs,
        batch_size=16,
        show_progress_bar=False,
    )

    reranked = []

    for hit, rerank_score in zip(hits, rerank_scores):
        row = dict(hit)

        # Se conserva para inspección y desempate
        row["base_score"] = float(hit.get("final_score", hit.get("score", 0.0)))
        row["rerank_score"] = float(rerank_score)
        row["final_rerank_score"] = float(rerank_score)

        reranked.append(row)

    reranked = sorted(
        reranked,
        key=lambda x: (x["final_rerank_score"], x["base_score"]),
        reverse=True,
    )

    for i, row in enumerate(reranked, start=1):
        row["rank"] = i

    return reranked

### 9.3 Restringir el reranking a ciertos casos

En este bloque se define cuándo merece la pena aplicar una segunda capa de ranking.

**Qué hace este bloque:**
- no aplica reranking a todas las queries
- evita rerank en preguntas ya muy guiadas por metadata estructural
- activa rerank sobre todo en preguntas de evidencia clínica
- da prioridad a queries que mencionan estudios concretos como `SECURE`, `PANDORA` o `NEPTUNO`

**Por qué se hace así:**
Si una query ya está bien resuelta con filtros como `section_code`, `subsection_title` o `formulation`, el reranker suele aportar poco.  
En cambio, en preguntas más abiertas sobre resultados o conclusiones de estudios sí puede ayudar a reordenar mejor los candidatos.

In [ ]:
def should_rerank(query):
    if infer_subsection(query) is not None:
        return False
    if infer_heading_leaf(query) is not None:
        return False
    if infer_formulation(query) is not None:
        return False

    q = norm_text(query)
    filename = infer_filename(query)

    if filename in {
        CANONICAL_STUDIES.get("secure"),
        CANONICAL_STUDIES.get("pandora"),
        CANONICAL_STUDIES.get("neptuno"),
    }:
        return True

    if re.search(r"estudio|ensayo|publicaci[oó]n|paper|art[ií]culo", q, re.I):
        return True

    if re.search(r"qué resultado|qué mostró|qué muestra|qué concluye|resultados?|conclus", q, re.I):
        return True

    return False


def search_chunks_reranked(query, top_k=RERANK_TOP_K, candidate_pool=RERANK_CANDIDATES):
    """
    Usa baseline por defecto.
    Solo aplica reranking cuando la query lo justifica.
    """
    base_hits = search_chunks(query, top_k=candidate_pool)

    if not should_rerank(query):
        return base_hits[:top_k]

    reranked_hits = rerank_hits(query, base_hits)
    return reranked_hits[:top_k]

### 9.4 Inspeccionar manualmente los resultados rerankeados

En este bloque se revisa qué devuelve el sistema después de aplicar reranking.

**Qué muestra:**
- posición final
- score del baseline
- score del reranker
- archivo
- sección
- subsección
- vista previa del chunk

**Por qué es útil:**
Permite comprobar si el reranker realmente mejora el orden de los candidatos o si está moviendo arriba resultados peores.

In [ ]:
def inspect_query_reranked(query, top_k=5, candidate_pool=RERANK_CANDIDATES, preview_chars=160):
    hits = search_chunks_reranked(query, top_k=top_k, candidate_pool=candidate_pool)

    df = pd.DataFrame([
        {
            "rank": i,
            "base_score": round(float(h.get("base_score", h.get("final_score", h.get("score", 0.0)))), 4),
            "rerank_score": round(float(h.get("rerank_score", h.get("final_rerank_score", 0.0))), 4),
            "final_rerank_score": round(float(h.get("final_rerank_score", h.get("rerank_score", 0.0))), 4),
            "filename": h.get("filename"),
            "section_code": h.get("section_code"),
            "subsection_title": h.get("subsection_title"),
            "preview": (h.get("text", "") or "")[:preview_chars].replace("\n", " "),
        }
        for i, h in enumerate(hits, start=1)
    ])

    display(df)
    return hits

In [ ]:
# probamos
inspect_query_reranked("¿Qué resultado mostró SECURE sobre eventos cardiovasculares mayores?", top_k=5)
inspect_query_reranked("¿Qué concluye PANDORA?", top_k=5)
inspect_query_reranked("¿Qué muestra NEPTUNO sobre persistencia terapéutica?", top_k=5)

### 9.5 Preparar la evaluación comparativa

En este bloque se define una función auxiliar para evaluar cualquier método de búsqueda sobre el mismo banco de queries.

**Qué hace este bloque:**
- reutiliza `EVAL_QUERIES` del apartado 8
- evalúa un método de búsqueda dado (`search_chunks` o `search_chunks_reranked`)
- calcula métricas comparables:
  - `hit@1`
  - `hit@3`
  - `hit@5`
  - `MRR`

**Objetivo:**
Poder comparar baseline y reranking en igualdad de condiciones.

In [ ]:
def evaluate_retrieval_with_method(eval_queries, search_fn, method_name, top_k=5):
    rows = []

    for spec in eval_queries:
        hits = search_fn(spec["query"], top_k=top_k)
        rr = first_relevant_rank(hits, spec)

        rows.append({
            "method": method_name,
            "id": spec["id"],
            "category": spec["category"],
            "query": spec["query"],
            "hit@1": int(rr == 1),
            "hit@3": int(rr is not None and rr <= 3),
            "hit@5": int(rr is not None and rr <= 5),
            "mrr": 0.0 if rr is None else round(1 / rr, 3),
            "first_rel_rank": rr,
            "top1_file": hits[0]["filename"] if hits else None,
            "top1_stage": hits[0].get("stage") if hits else None,
            "top1_section": hits[0].get("section_code") if hits else None,
            "top1_subsection": hits[0].get("subsection_title") if hits else None,
        })

    detail_df = pd.DataFrame(rows)

    summary_df = pd.DataFrame([{
        "method": method_name,
        "n_queries": len(detail_df),
        "hit@1": round(detail_df["hit@1"].mean(), 3),
        "hit@3": round(detail_df["hit@3"].mean(), 3),
        "hit@5": round(detail_df["hit@5"].mean(), 3),
        "mrr": round(detail_df["mrr"].mean(), 3),
    }])

    return detail_df, summary_df

### 9.6 Comparar baseline y reranking

Aquí se ejecuta la evaluación con los dos métodos:

- baseline: `search_chunks`
- reranking: `search_chunks_reranked`

**Qué muestra este bloque:**
- una tabla resumen con métricas agregadas
- una tabla detallada por query
- una comparación directa entre ambos métodos

**Objetivo:**
Ver si el reranking mejora realmente el retrieval o si no aporta valor medible.

In [ ]:
baseline_eval_df, baseline_summary_df = evaluate_retrieval_with_method(
    EVAL_QUERIES,
    search_fn=search_chunks,
    method_name="baseline",
    top_k=5,
)

rerank_eval_df, rerank_summary_df = evaluate_retrieval_with_method(
    EVAL_QUERIES,
    search_fn=search_chunks_reranked,
    method_name="rerank_guarded",
    top_k=5,
)

comparison_summary_df = pd.concat(
    [baseline_summary_df, rerank_summary_df],
    ignore_index=True,
)

comparison_eval_df = baseline_eval_df.merge(
    rerank_eval_df,
    on=["id", "category", "query"],
    suffixes=("_baseline", "_rerank"),
)

display(comparison_summary_df)
display(comparison_eval_df)

### 9.7 Detectar cambios reales introducidos por el reranking

En este bloque se filtran solo las queries en las que el reranking cambia algo relevante frente al baseline.

**Qué revisa:**
- si cambia el `top1`
- si cambia la posición del primer resultado relevante
- si mejora o empeora `hit@1`

**Objetivo:**
No revisar todas las queries, sino centrarse en los casos donde el reranking realmente modifica el comportamiento del sistema.

In [ ]:
comparison_changes_df = comparison_eval_df.copy()

comparison_changes_df["top1_changed"] = (
    comparison_changes_df["top1_file_baseline"] != comparison_changes_df["top1_file_rerank"]
)

comparison_changes_df["rank_changed"] = (
    comparison_changes_df["first_rel_rank_baseline"] != comparison_changes_df["first_rel_rank_rerank"]
)

comparison_changes_df["hit1_changed"] = (
    comparison_changes_df["hit@1_baseline"] != comparison_changes_df["hit@1_rerank"]
)

changed_queries_df = comparison_changes_df[
    comparison_changes_df["top1_changed"] |
    comparison_changes_df["rank_changed"] |
    comparison_changes_df["hit1_changed"]
].copy()

display(
    changed_queries_df[
        [
            "id",
            "category",
            "query",
            "top1_file_baseline",
            "top1_file_rerank",
            "first_rel_rank_baseline",
            "first_rel_rank_rerank",
            "hit@1_baseline",
            "hit@1_rerank",
        ]
    ].sort_values(["category", "id"])
)

### 9.8 Conclusión del experimento de reranking

En este bloque se resume el impacto del reranking sobre las métricas principales.

**Qué hace este bloque:**
- compara baseline y reranking en:
  - `hit@1`
  - `hit@3`
  - `hit@5`
  - `MRR`
- calcula la diferencia entre ambos métodos
- deja una base objetiva para decidir si el reranking merece la pena

**Importante:**
La decisión no debe basarse solo en ejemplos aislados, sino en el comportamiento agregado sobre el banco de evaluación.

In [ ]:
rerank_vs_baseline_df = comparison_summary_df.set_index("method").copy()

delta_metrics_df = pd.DataFrame([{
    "metric": metric,
    "baseline": float(rerank_vs_baseline_df.loc["baseline", metric]),
    "rerank_guarded": float(rerank_vs_baseline_df.loc["rerank_guarded", metric]),
    "delta": round(
        float(rerank_vs_baseline_df.loc["rerank_guarded", metric]) -
        float(rerank_vs_baseline_df.loc["baseline", metric]),
        3
    ),
} for metric in ["hit@1", "hit@3", "hit@5", "mrr"]])

display(delta_metrics_df)

if (delta_metrics_df["delta"] > 0).any():
    print("El reranking aporta alguna mejora en al menos una métrica.")
else:
    print("Con esta configuración, el reranking no aporta mejora medible frente al baseline.")

### Comentario final

La versión actual del sistema ya tiene una base bastante más sólida que al inicio: el chunking está mejor estructurado, la indexación funciona correctamente, la evaluación del retrieval es más útil y el experimento de reranking ya permite comparar resultados de forma objetiva.

A partir de lo visto en este notebook, las conclusiones principales son estas:

- El **retrieval baseline** ya resuelve razonablemente bien una parte importante de las queries si la pregunta está bien guiada por estructura (`section_code`, `subsection_title`, `formulation`, `filename`).
- El **reranking controlado** aporta una **mejora pequeña pero medible** en el banco de evaluación actual, por lo que no debe descartarse, aunque tampoco conviene asumir todavía que será útil en todos los escenarios.
- El banco de evaluación sigue siendo **limitado**. Sirve para comparar versiones del pipeline, pero todavía no es suficiente para afirmar robustez en un contexto médico exigente.
- La arquitectura actual está optimizada sobre todo para **recuperar chunks relevantes**, pero aún no está cerrada como sistema de **respuesta clínica segura**.

Como líneas prioritarias de mejora futura, tendría sentido trabajar en este orden:

1. **Ampliar la evaluación**
   - añadir más queries
   - cubrir mejor indicación, contraindicaciones, poblaciones especiales, seguridad, eficacia y coste-efectividad
   - incluir más casos difíciles y ambiguos

2. **Reforzar la validación del reranking**
   - revisar en qué queries mejora realmente
   - comprobar si la mejora se mantiene al ampliar el banco de evaluación
   - decidir después si merece la pena dejarlo en producción

3. **Reducir todavía más la fragilidad del retrieval**
   - seguir limitando heurísticas demasiado específicas del corpus
   - depender más de metadata estable y menos de reglas manuales hechas a medida

4. **Mejorar la capa de respuesta final**
   - introducir abstención cuando no haya evidencia suficiente
   - obligar a responder con citas de documento/sección/página
   - añadir alguna señal de confianza o criterio de no respuesta

5. **Mejorar la deduplicación y el control de redundancia**
   - ahora la deduplicación entre páginas sigue siendo exacta
   - más adelante podría ser útil una deduplicación aproximada para PDFs con contenido casi repetido

En resumen, el notebook ya está en un punto razonable para seguir iterando con criterio. El siguiente salto de calidad no pasa tanto por añadir más complejidad, sino por **evaluar mejor, simplificar donde sea posible y endurecer la seguridad de la respuesta final**.

## 10 Generación de respuestas con RAG

En este bloque se añade la capa de generación final del sistema. Hasta ahora el pipeline recupera y reranquea chunks, pero todavía falta convertir ese contexto en una respuesta final para el usuario.

**Qué hace este bloque:**

- toma los chunks devueltos por `search_chunks_reranked`
- construye un contexto unificado a partir de esos resultados
- prepara un prompt restringido
- llama a un modelo generador
- devuelve una respuesta final junto con el contexto usado

**Por qué es importante:**  
Este bloque convierte el sistema en un RAG completo. Además, deja preparada la salida que después podrá evaluarse con un juez como `Prometheus` o `JudgeLM`.

### 10.1 Configuración base de la generación

Este bloque define algunos parámetros base para la generación y fija el comportamiento por defecto.

**Qué hace este bloque:**

- fija el `top_k` que se usará para construir el contexto
- deja preparado un modo de generación determinista
- separa claramente retrieval y generación

**Por qué es importante:**  
Antes de evaluar respuestas, conviene fijar una forma estable de generarlas para que las comparaciones sean reproducibles.

In [ ]:
# 10.1
# Parámetros base para la generación

GEN_TOP_K = 3          # número de chunks que se usarán como contexto
GEN_TEMPERATURE = 0.0  # respuesta lo más determinista posible

### 10.2 Construir el contexto a partir de los hits recuperados

En este bloque se transforma la salida del retrieval en un bloque de contexto legible y trazable.

**Qué hace este bloque:**

- recorre los hits recuperados
- concatena los chunks en un único contexto
- conserva metadatos útiles como archivo, heading y página
- separa claramente cada fragmento

**Por qué es importante:**  
Así el modelo generador recibe evidencia estructurada y no una mezcla opaca de texto.

In [ ]:
# 10.2
# Convierte los hits recuperados en un contexto único

def build_context(hits):
    blocks = []

    for i, h in enumerate(hits, start=1):
        source = (
            f"[CTX {i}] "
            f"{h.get('filename', 'NA')} | "
            f"{h.get('heading_leaf', 'NA')} | "
            f"p.{h.get('page_start', '?')}"
        )

        text = h.get("text", "").strip()
        blocks.append(f"{source}\n{text}")

    return "\n\n".join(blocks)

### 10.3 Definir un prompt de generación restringida

En este bloque se construye el prompt que verá el modelo generador.

**Qué hace este bloque:**

- le indica al modelo que responda solo con el contexto recuperado
- le pide que no invente información externa
- le obliga a reconocer cuando el contexto no es suficiente

**Por qué es importante:**  
En un RAG médico interesa minimizar alucinaciones y mantener la respuesta pegada a la evidencia recuperada.

In [ ]:
# 10.3
# Prompt para obligar al modelo a responder solo con el contexto

def make_rag_prompt(query, context):
    return f"""
Responde SOLO con la información del contexto recuperado.
Si el contexto no es suficiente, di claramente:
"No lo puedo confirmar con el contexto recuperado".

Pregunta:
{query}

Contexto:
{context}

Respuesta:
""".strip()

### 10.4 Conectar el modelo generador

En este bloque se define la llamada al LLM que producirá la respuesta final.

**Qué hace este bloque:**

- recibe el prompt final
- llama al modelo generador
- devuelve la respuesta en texto

**Por qué es importante:**  
Aquí aparece por primera vez una `answer` real que luego podrá evaluarse automáticamente.

In [ ]:
# 10.4
# Sustituye esta función por tu llamada real al modelo

def call_generator(prompt):
    # TODO: conecta aquí tu modelo real
    # Ejemplo orientativo:
    #
    # response = client.chat.completions.create(
    #     model="tu-modelo",
    #     messages=[{"role": "user", "content": prompt}],
    #     temperature=GEN_TEMPERATURE
    # )
    # return response.choices[0].message.content.strip()

    return "TODO: conecta aquí tu modelo generador"

### 10.5 Crear una función completa de respuesta RAG

En este bloque se empaqueta todo el flujo de generación en una sola función.

**Qué hace este bloque:**

- ejecuta `search_chunks_reranked`
- construye el contexto
- genera el prompt final
- llama al modelo
- devuelve:
  - `query`
  - `hits`
  - `context`
  - `prompt`
  - `answer`

**Por qué es importante:**  
Deja una única función reutilizable para inspección manual y para evaluación automática posterior.

In [ ]:
# 10.5
# Función completa de RAG: retrieval + contexto + prompt + generación

def answer_with_rag(query, top_k=GEN_TOP_K):
    # recupera los mejores chunks con el pipeline actual
    hits = search_chunks_reranked(query, top_k=top_k)

    # construye el contexto que verá el generador
    context = build_context(hits)

    # prepara el prompt final
    prompt = make_rag_prompt(query, context)

    # genera la respuesta
    answer = call_generator(prompt)

    # devuelve todo lo necesario para inspección y evaluación
    return {
        "query": query,
        "hits": hits,
        "context": context,
        "prompt": prompt,
        "answer": answer,
    }

### 10.6 Inspección manual de una respuesta generada

Este bloque permite revisar visualmente qué devuelve ya el sistema RAG completo para una pregunta concreta.

**Qué muestra este bloque:**

- la pregunta
- la respuesta generada
- el contexto usado
- los hits recuperados

**Por qué es útil:**  
Antes de evaluar automáticamente, conviene comprobar manualmente si la respuesta realmente está apoyada en los chunks correctos.

In [ ]:
# 10.6
# Prueba manual rápida de generación

query = "¿Qué contraindicaciones tiene MedProduct?"
result = answer_with_rag(query, top_k=3)

print("QUERY:\n")
print(result["query"])

print("\nRESPUESTA:\n")
print(result["answer"])

print("\nCONTEXTO USADO:\n")
print(result["context"][:3000])  # recorte para no imprimir demasiado

### 10.7 Generar respuestas en lote para un banco de evaluación

En este bloque se define una utilidad para generar respuestas sobre varias queries de evaluación.

**Qué hace este bloque:**

- recorre un conjunto de preguntas
- llama a `answer_with_rag` para cada una
- guarda pregunta, respuesta y contexto
- construye un dataframe con los resultados

**Por qué es importante:**  
Deja preparado el material que luego se podrá pasar a `Prometheus` o `JudgeLM`.

In [ ]:
# 10.7
# Genera respuestas en lote sobre un conjunto de queries

import pandas as pd

def generate_answers_df(eval_queries, top_k=GEN_TOP_K):
    rows = []

    for spec in eval_queries:
        query = spec["query"]
        out = answer_with_rag(query, top_k=top_k)

        rows.append({
            "id": spec.get("id"),
            "category": spec.get("category"),
            "query": query,
            "answer": out["answer"],
            "context": out["context"],
            "n_hits": len(out["hits"]),
            "top1_file": out["hits"][0].get("filename") if out["hits"] else None,
            "top1_heading": out["hits"][0].get("heading_leaf") if out["hits"] else None,
        })

    return pd.DataFrame(rows)

### 10.8 Ejecutar la generación y guardar resultados

En este bloque se ejecuta la generación sobre el banco de evaluación y se guardan los resultados para análisis posterior.

**Qué hace este bloque:**

- genera respuestas para todas las queries
- construye un dataframe reproducible
- deja una copia estable para la evaluación posterior

**Por qué es importante:**  
Así el punto 11 podrá centrarse solo en evaluar respuestas, sin regenerarlas cada vez.

In [ ]:
# 10.8
# Ejecuta la generación en lote
# Aquí reutilizamos EVAL_QUERIES del punto 8

df_answers = generate_answers_df(EVAL_QUERIES, top_k=GEN_TOP_K)
df_answers.head()

## 11 Evaluación de respuestas con Prometheus

En este bloque se añade una capa de evaluación automática sobre las respuestas generadas por el sistema RAG.

**Qué hace este bloque:**

- toma la `query`, la `answer` y el `context` generados en el punto 10
- construye una instrucción de evaluación para el juez
- define una rúbrica de fidelidad al contexto
- puntúa cada respuesta con un score de 1 a 5
- guarda también el feedback textual del juez

**Por qué es importante:**  
Hasta el punto 10 el sistema ya genera respuestas. En este punto medimos si esas respuestas son fieles al contexto recuperado y si evitan alucinaciones.

### 11.1 Instalar e importar Prometheus

Este bloque prepara la librería de evaluación.

**Qué hace este bloque:**

- instala `prometheus-eval`
- importa la clase `PrometheusEval`
- importa un backend de inferencia para el juez

**Por qué es importante:**  
Necesitamos un modelo juez para puntuar automáticamente las respuestas del RAG.

In [ ]:
from prometheus_eval import PrometheusEval
from prometheus_eval.prompts import ABSOLUTE_PROMPT
from prometheus_eval.litellm import LiteLLM

### 11.2 Conectar el modelo juez

En este bloque se define qué modelo actuará como juez.

**Qué hace este bloque:**

- crea el backend del modelo juez
- inicializa `PrometheusEval`
- deja listo el evaluador para puntuar respuestas

**Por qué es importante:**  
A partir de aquí ya podemos enviar respuestas del RAG al juez automático.

In [ ]:
# 11.2
# Cambia el identificador por el endpoint/modelo que vayas a usar de verdad

judge_model = LiteLLM("openai/prometheus-eval/prometheus-7b-v2.0")
judge = PrometheusEval(
    model=judge_model,
    absolute_grade_template=ABSOLUTE_PROMPT,
)

### 11.3 Definir la rúbrica de evaluación

En este bloque se define qué significa una buena respuesta en este RAG médico.

**Qué evalúa esta rúbrica:**

- fidelidad al contexto recuperado
- ausencia de alucinaciones
- prudencia al responder
- reconocimiento explícito de límites cuando el contexto no basta

**Por qué es importante:**  
No queremos puntuar respuestas por “sonar bien”, sino por estar realmente apoyadas en los documentos recuperados.

In [ ]:
# 11.3
# Rúbrica simple de 1 a 5 para fidelidad al contexto en RAG médico

GROUNDING_RUBRIC = """
[Criteria]
Evalúa la fidelidad de una respuesta médica al contexto recuperado por un sistema RAG.

[Score 1]
La respuesta inventa información, contradice el contexto o hace afirmaciones médicas no apoyadas.

[Score 2]
La respuesta usa algo del contexto, pero mezcla varios datos no sustentados o inferencias arriesgadas.

[Score 3]
La respuesta es bastante compatible con el contexto, pero añade alguna inferencia débil u omite una limitación importante.

[Score 4]
La respuesta es fiel al contexto y no inventa hechos relevantes, aunque puede tener pequeñas imprecisiones.

[Score 5]
La respuesta es totalmente fiel al contexto recuperado, no introduce afirmaciones externas y expresa claramente los límites de la evidencia.
""".strip()

### 11.4 Construir la instrucción que verá el juez

En este bloque se transforma cada caso del RAG en una instrucción evaluable por Prometheus.

**Qué hace este bloque:**

- incluye la pregunta original
- incluye el contexto recuperado
- deja la respuesta aparte para que el juez la puntúe
- centra la evaluación en grounding y seguridad

**Por qué es importante:**  
El juez necesita ver la pregunta y el contexto para decidir si la respuesta está realmente justificada.

In [ ]:
# 11.4
# Convierte query + contexto en la instrucción que recibirá el juez

def make_judge_instruction(query, context):
    return f"""
Eres un evaluador de un sistema RAG para documentación médica y regulatoria.

Evalúa la respuesta SOLO en función de su fidelidad al contexto recuperado.
No premies conocimiento externo.
Penaliza:
- alucinaciones
- extrapolaciones no apoyadas
- afirmaciones médicas fuera del contexto
- omisión de límites cuando el contexto no basta

Pregunta:
{query}

Contexto recuperado:
{context}
""".strip()

### 11.5 Evaluar una respuesta individual

En este bloque se puntúa una sola respuesta generada por el sistema.

**Qué hace este bloque:**

- construye la instrucción de evaluación
- envía la respuesta al juez
- recibe `feedback` y `score`
- devuelve el resultado en un formato reutilizable

**Por qué es importante:**  
Permite inspeccionar casos concretos antes de lanzar la evaluación en lote.

In [ ]:
# 11.5
# Evalúa una sola respuesta del RAG con Prometheus

def judge_one_answer(query, answer, context, reference_answer=None):
    instruction = make_judge_instruction(query, context)

    feedback, score = judge.single_absolute_grade(
        instruction=instruction,
        response=answer,
        rubric=GROUNDING_RUBRIC,
        reference_answer=reference_answer,
    )

    return {
        "feedback_grounding": feedback,
        "score_grounding": score,
    }

### 11.6 Probar la evaluación sobre un ejemplo

Este bloque permite revisar manualmente cómo puntúa el juez una respuesta concreta.

**Qué muestra este bloque:**

- la pregunta
- la respuesta generada
- el score del juez
- el feedback textual

**Por qué es útil:**  
Sirve para comprobar si la rúbrica está alineada con lo que quieres medir.

In [ ]:
#################

judge_model = VLLM(model="prometheus-eval/prometheus-7b-v2.0")
judge = PrometheusEval(
    model=judge_model,
    absolute_grade_template=ABSOLUTE_PROMPT,
)

In [ ]:
# 11.6
# Prueba rápida sobre un ejemplo del punto 10

example = df_answers.iloc[0]

judge_result = judge_one_answer(
    query=example["query"],
    answer=example["answer"],
    context=example["context"],
)

print("QUERY:\n", example["query"])
print("\nANSWER:\n", example["answer"])
print("\nSCORE:\n", judge_result["score_grounding"])
print("\nFEEDBACK:\n", judge_result["feedback_grounding"])

### 11.7 Evaluar todas las respuestas generadas

En este bloque se aplica el juez a todas las respuestas del dataframe generado en el punto 10.

**Qué hace este bloque:**

- recorre `df_answers`
- puntúa cada respuesta con Prometheus
- guarda score y feedback
- devuelve un dataframe enriquecido con la evaluación

**Por qué es importante:**  
Permite medir la calidad del sistema de forma sistemática y no solo caso a caso.

In [ ]:
# 11.7
# Evaluación en lote sobre todas las respuestas generadas

def evaluate_answers_with_judge(df_answers):
    rows = []

    for _, row in df_answers.iterrows():
        result = judge_one_answer(
            query=row["query"],
            answer=row["answer"],
            context=row["context"],
        )

        rows.append({
            "id": row.get("id"),
            "category": row.get("category"),
            "query": row["query"],
            "answer": row["answer"],
            "score_grounding": result["score_grounding"],
            "feedback_grounding": result["feedback_grounding"],
            "top1_file": row.get("top1_file"),
            "top1_heading": row.get("top1_heading"),
        })

    return pd.DataFrame(rows)

### 11.8 Ejecutar la evaluación automática

En este bloque se lanza la evaluación sobre todo el banco de respuestas.

**Qué hace este bloque:**

- crea un nuevo dataframe con los resultados del juez
- deja cada respuesta con su score y feedback
- prepara el material para análisis agregado

**Por qué es importante:**  
A partir de aquí ya puedes comparar versiones del sistema con una métrica de calidad de respuesta.

In [ ]:
# 11.8
# Lanza la evaluación completa

df_judged = evaluate_answers_with_judge(df_answers)
df_judged.head()

### 11.9 Resumir resultados globales y por categoría

En este bloque se agregan los resultados para tener una visión global del sistema.

**Qué hace este bloque:**

- calcula la media global del score
- calcula medias por categoría
- permite detectar dónde falla más el sistema

**Por qué es importante:**  
No basta con ver ejemplos sueltos. Necesitamos una vista agregada de rendimiento.

In [ ]:
# 11.9
# Resumen global y por categoría

overall_judge_df = pd.DataFrame([{
    "metric": "mean_grounding_score",
    "value": df_judged["score_grounding"].mean()
}])

by_category_judge_df = (
    df_judged.groupby("category", dropna=False)["score_grounding"]
    .mean()
    .reset_index(name="mean_grounding_score")
    .sort_values("mean_grounding_score", ascending=False)
)

overall_judge_df, by_category_judge_df

### 11.10 Guardar los resultados de la evaluación

En este bloque se guardan los resultados del juez para compararlos después con otras variantes del sistema.

**Qué hace este bloque:**

- guarda el dataframe de respuestas evaluadas
- conserva métricas globales y por categoría
- deja una copia reproducible del experimento

**Por qué es importante:**  
Así podrás comparar más adelante baseline vs reranking, distintos prompts o distintos modelos generadores.

In [ ]:
# 11.10
# Guardar resultados de evaluación

df_judged.to_csv("rag_judged_answers.csv", index=False)
overall_judge_df.to_csv("rag_judge_overall.csv", index=False)
by_category_judge_df.to_csv("rag_judge_by_category.csv", index=False)